In [2]:
# ============================================================
# SIGNALDRAW - BIOMEDICAL SIGNAL VISUALIZATION
# ============================================================
#
# INSTALL:
# pip install anywidget traitlets numpy plotly
#
# ============================================================

import anywidget
import traitlets
import numpy as np


class SignalDraw(anywidget.AnyWidget):

    _esm = r"""

export async function render({ model, el }) {

    // Load Plotly
    await new Promise(resolve => {
        if (window.Plotly) {
            resolve();
        } else {
            const script = document.createElement("script");
            script.src = "https://cdn.plot.ly/plotly-2.35.2.min.js";
            script.onload = () => resolve();
            document.head.appendChild(script);
        }
    });

    const Plotly = window.Plotly;

    // Colors
    const COLORS = ["#0066ff", "#ff0000", "#00aa00"];
    const BG = "#e8e8e8";
    const WHITE = "#ffffff";
    const BORDER = "#cccccc";

    // Root
    const root = document.createElement("div");
    root.style.display = "flex";
    root.style.flexDirection = "column";
    root.style.width = "100%";
    root.style.background = BG;
    root.style.padding = "12px";
    el.appendChild(root);

    // Topbar
    const topbar = document.createElement("div");
    topbar.style.display = "flex";
    topbar.style.gap = "15px";
    topbar.style.alignItems = "center";
    topbar.style.padding = "12px";
    topbar.style.background = WHITE;
    topbar.style.borderRadius = "6px";
    topbar.style.marginBottom = "12px";
    root.appendChild(topbar);

    const title = document.createElement("h1");
    title.innerText = "SignalDraw";
    title.style.margin = "0";
    title.style.fontSize = "24px";
    title.style.color = "#0066ff";
    title.style.marginRight = "auto";
    topbar.appendChild(title);

    const label = document.createElement("label");
    label.innerText = "#Signals: ";
    topbar.appendChild(label);

    const select = document.createElement("select");
    select.style.padding = "6px 10px";
    for(let i=1; i<=5; i++) {
        const opt = document.createElement("option");
        opt.value = i;
        opt.text = i;
        if(i===2) opt.selected = true;
        select.appendChild(opt);
    }
    topbar.appendChild(select);

    const btnGenerate = document.createElement("button");
    btnGenerate.innerText = "GENERATE";
    btnGenerate.style.padding = "8px 16px";
    btnGenerate.style.background = "#0066ff";
    btnGenerate.style.color = "white";
    btnGenerate.style.border = "none";
    btnGenerate.style.borderRadius = "4px";
    btnGenerate.style.cursor = "pointer";
    topbar.appendChild(btnGenerate);

    // Main container
    const main = document.createElement("div");
    main.style.display = "flex";
    main.style.gap = "12px";
    main.style.height = "700px";
    root.appendChild(main);

    // Panels
    const leftPanel = document.createElement("div");
    leftPanel.style.width = "28%";
    leftPanel.style.display = "flex";
    leftPanel.style.flexDirection = "column";
    leftPanel.style.gap = "10px";
    leftPanel.style.overflowY = "auto";
    main.appendChild(leftPanel);

    const centerPanel = document.createElement("div");
    centerPanel.style.width = "36%";
    centerPanel.style.display = "flex";
    centerPanel.style.flexDirection = "column";
    centerPanel.style.gap = "10px";
    centerPanel.style.overflowY = "auto";
    main.appendChild(centerPanel);

    const rightPanel = document.createElement("div");
    rightPanel.style.width = "36%";
    rightPanel.style.display = "flex";
    rightPanel.style.flexDirection = "column";
    rightPanel.style.gap = "10px";
    main.appendChild(rightPanel);

    // Plot containers in right panel
    const sumContainer = document.createElement("div");
    sumContainer.style.height = "48%";
    sumContainer.style.background = WHITE;
    sumContainer.style.borderRadius = "6px";
    sumContainer.style.border = "1px solid " + BORDER;
    rightPanel.appendChild(sumContainer);

    const fftContainer = document.createElement("div");
    fftContainer.style.height = "48%";
    fftContainer.style.background = WHITE;
    fftContainer.style.borderRadius = "6px";
    fftContainer.style.border = "1px solid " + BORDER;
    rightPanel.appendChild(fftContainer);

    // State
    let signals = [];

    // Build UI
    function buildUI() {
        leftPanel.innerHTML = "";
        centerPanel.innerHTML = "";
        signals = [];

        const n = parseInt(select.value);

        for(let k = 0; k < n; k++) {
            // Left: Config
            const card = document.createElement("div");
            card.style.background = WHITE;
            card.style.padding = "10px";
            card.style.borderRadius = "6px";
            card.style.border = "2px solid " + COLORS[k];
            card.style.borderLeft = "5px solid " + COLORS[k];

            const cardTitle = document.createElement("h3");
            cardTitle.innerText = "Signal " + (k+1);
            cardTitle.style.margin = "0 0 8px 0";
            cardTitle.style.fontSize = "13px";
            cardTitle.style.color = COLORS[k];
            card.appendChild(cardTitle);

            const fsLabel = document.createElement("label");
            fsLabel.innerText = "Fs (Hz)";
            fsLabel.style.fontSize = "11px";
            fsLabel.style.fontWeight = "bold";
            card.appendChild(fsLabel);
            const fsInput = document.createElement("input");
            fsInput.type = "number";
            fsInput.value = 500;
            fsInput.style.width = "100%";
            fsInput.style.padding = "4px";
            fsInput.style.marginBottom = "6px";
            fsInput.style.boxSizing = "border-box";
            card.appendChild(fsInput);

            const freqLabel = document.createElement("label");
            freqLabel.innerText = "Frequency (Hz)";
            freqLabel.style.fontSize = "11px";
            freqLabel.style.fontWeight = "bold";
            card.appendChild(freqLabel);
            const freqInput = document.createElement("input");
            freqInput.type = "number";
            freqInput.value = 5 + k;
            freqInput.style.width = "100%";
            freqInput.style.padding = "4px";
            freqInput.style.marginBottom = "6px";
            freqInput.style.boxSizing = "border-box";
            card.appendChild(freqInput);

            const ampLabel = document.createElement("label");
            ampLabel.innerText = "Amplitude";
            ampLabel.style.fontSize = "11px";
            ampLabel.style.fontWeight = "bold";
            card.appendChild(ampLabel);
            const ampSlider = document.createElement("input");
            ampSlider.type = "range";
            ampSlider.min = 0.1;
            ampSlider.max = 5;
            ampSlider.step = 0.1;
            ampSlider.value = 1;
            ampSlider.style.width = "100%";
            card.appendChild(ampSlider);

            const phaseLabel = document.createElement("label");
            phaseLabel.innerText = "Phase (rad)";
            phaseLabel.style.fontSize = "11px";
            phaseLabel.style.fontWeight = "bold";
            card.appendChild(phaseLabel);
            const phaseInput = document.createElement("input");
            phaseInput.type = "number";
            phaseInput.value = 0;
            phaseInput.style.width = "100%";
            phaseInput.style.padding = "4px";
            phaseInput.style.boxSizing = "border-box";
            card.appendChild(phaseInput);

            leftPanel.appendChild(card);

            // Center: Plot
            const plotDiv = document.createElement("div");
            plotDiv.style.height = "210px";
            plotDiv.style.background = WHITE;
            plotDiv.style.borderRadius = "6px";
            plotDiv.style.border = "1px solid " + BORDER;
            centerPanel.appendChild(plotDiv);

            signals.push({
                fs: fsInput,
                freq: freqInput,
                amp: ampSlider,
                phase: phaseInput,
                plot: plotDiv,
                color: COLORS[k],
                idx: k
            });
        }
    }

    // FFT
    function fft(signal) {
        const N = signal.length;
        let mags = [];
        let freqs = [];
        for(let k=0; k<Math.min(N/2, 200); k++) {
            let re = 0, im = 0;
            for(let n=0; n<N; n++) {
                const angle = 2 * Math.PI * k * n / N;
                re += signal[n] * Math.cos(angle);
                im -= signal[n] * Math.sin(angle);
            }
            mags.push(Math.sqrt(re*re + im*im) / N);
            freqs.push(k);
        }
        return { freqs, mags };
    }

    // Generate plots
    function generate() {
        console.log("GENERATING...");
        let total = [];
        let time = [];

        signals.forEach(sig => {
            const fs = parseFloat(sig.fs.value) || 500;
            const freq = parseFloat(sig.freq.value) || 5;
            const amp = parseFloat(sig.amp.value) || 1;
            const phase = parseFloat(sig.phase.value) || 0;

            const N = Math.round(fs * 2);
            let vals = [];
            let t = [];

            for(let i=0; i<N; i++) {
                const tt = i / fs;
                const y = amp * Math.sin(2 * Math.PI * freq * tt + phase);
                vals.push(y);
                t.push(tt);
            }

            time = t;

            if(total.length === 0) {
                total = [...vals];
            } else {
                total = total.map((v, i) => v + (vals[i] || 0));
            }

            // Plot signal
            try {
                Plotly.newPlot(sig.plot, [{
                    x: t,
                    y: vals,
                    mode: "lines",
                    line: { color: sig.color, width: 2 }
                }], {
                    title: "Signal " + (sig.idx + 1),
                    margin: { l: 35, r: 15, t: 25, b: 25 },
                    paper_bgcolor: WHITE,
                    plot_bgcolor: "#fafafa",
                    font: { color: "#333", size: 9 },
                    showlegend: false
                }, { responsive: true, displayModeBar: false });
                console.log("Signal " + (sig.idx + 1) + " plotted");
            } catch(e) {
                console.error("Error signal", sig.idx, e);
            }
        });

        // Sum
        if(time.length > 0) {
            try {
                Plotly.newPlot(sumContainer, [{
                    x: time,
                    y: total,
                    mode: "lines",
                    line: { color: "#0066ff", width: 2 }
                }], {
                    title: "Summed Signal",
                    margin: { l: 40, r: 15, t: 30, b: 30 },
                    paper_bgcolor: WHITE,
                    plot_bgcolor: "#fafafa",
                    font: { color: "#333" },
                    showlegend: false
                }, { responsive: true, displayModeBar: false });
                console.log("Sum plotted");

                // FFT
                const f = fft(total);
                Plotly.newPlot(fftContainer, [{
                    x: f.freqs,
                    y: f.mags,
                    mode: "lines",
                    line: { color: "#00aa00", width: 2 }
                }], {
                    title: "FFT Spectrum",
                    margin: { l: 40, r: 15, t: 30, b: 30 },
                    paper_bgcolor: WHITE,
                    plot_bgcolor: "#fafafa",
                    font: { color: "#333" },
                    showlegend: false
                }, { responsive: true, displayModeBar: false });
                console.log("FFT plotted");

                model.set("signal", total);
                model.set("time", time);
                model.save_changes();
            } catch(e) {
                console.error("Error sum/fft", e);
            }
        }
    }

    // Events
    select.addEventListener("change", () => {
        buildUI();
        setTimeout(generate, 100);
    });

    btnGenerate.addEventListener("click", generate);

    // Init
    buildUI();
    setTimeout(generate, 200);
}
"""

    signal = traitlets.List([]).tag(sync=True)
    time = traitlets.List([]).tag(sync=True)

    @property
    def signal_numpy(self):
        return np.array(self.signal)

    @property
    def time_numpy(self):
        return np.array(self.time)


ui = SignalDraw()
ui


In [3]:
import anywidget
import traitlets
import numpy as np


class SignalDraw(anywidget.AnyWidget):

    signals = traitlets.List([]).tag(sync=True)

    fs = traitlets.List([]).tag(sync=True)

    _esm = r"""

export function render({ model, el }) {

    // =====================================================
    // COLORS
    // =====================================================

    const COLORS = [
        "#0066ff",
        "#ff0000",
        "#00aa00",
        "#ff9900",
        "#9900ff"
    ];

    // =====================================================
    // ROOT
    // =====================================================

    const root = document.createElement("div");

    root.style.width = "100%";
    root.style.maxWidth = "1800px";
    root.style.background = "#efefef";
    root.style.padding = "8px";
    root.style.fontFamily = "Arial";
    root.style.boxSizing = "border-box";

    el.appendChild(root);

    // =====================================================
    // TOPBAR
    // =====================================================

    const topbar = document.createElement("div");

    topbar.style.display = "flex";
    topbar.style.alignItems = "center";
    topbar.style.gap = "12px";
    topbar.style.background = "white";
    topbar.style.padding = "8px";
    topbar.style.borderRadius = "6px";
    topbar.style.marginBottom = "8px";

    root.appendChild(topbar);

    // =====================================================
    // TITLE
    // =====================================================

    const title = document.createElement("h2");

    title.innerText = "SignalDraw";

    title.style.margin = "0";
    title.style.color = "#0066ff";
    title.style.marginRight = "auto";

    topbar.appendChild(title);

    // =====================================================
    // #SIGNALS
    // =====================================================

    const label =
        document.createElement("label");

    label.innerText = "#Signals";

    topbar.appendChild(label);

    const select =
        document.createElement("select");

    for(let i=1;i<=5;i++) {

        const op =
            document.createElement("option");

        op.value = i;
        op.text = i;

        if(i===1) op.selected = true;

        select.appendChild(op);
    }

    topbar.appendChild(select);

    // =====================================================
    // GENERATE BUTTON
    // =====================================================

    const btn =
        document.createElement("button");

    btn.innerText = "GENERATE";

    btn.style.padding = "6px 12px";
    btn.style.background = "#0066ff";
    btn.style.color = "white";
    btn.style.border = "none";
    btn.style.borderRadius = "5px";
    btn.style.cursor = "pointer";

    topbar.appendChild(btn);

    // =====================================================
    // RESET BUTTON
    // =====================================================

    const resetBtn =
        document.createElement("button");

    resetBtn.innerText = "RESET";

    resetBtn.style.padding = "6px 12px";
    resetBtn.style.background = "#ff4444";
    resetBtn.style.color = "white";
    resetBtn.style.border = "none";
    resetBtn.style.borderRadius = "5px";
    resetBtn.style.cursor = "pointer";

    topbar.appendChild(resetBtn);

    // =====================================================
    // MAIN
    // =====================================================

    const main = document.createElement("div");

    main.style.display = "flex";
    main.style.gap = "8px";

    root.appendChild(main);

    // =====================================================
    // LEFT PANEL
    // =====================================================

    const left =
        document.createElement("div");

    left.style.width = "25%";
    left.style.display = "flex";
    left.style.flexDirection = "column";
    left.style.gap = "8px";
    left.style.maxHeight = "1400px";
    left.style.overflowY = "auto";

    main.appendChild(left);

    // =====================================================
    // CENTER PANEL
    // =====================================================

    const center =
        document.createElement("div");

    center.style.width = "35%";
    center.style.display = "flex";
    center.style.flexDirection = "column";
    center.style.gap = "8px";

    main.appendChild(center);

    // =====================================================
    // RIGHT PANEL
    // =====================================================

    const right =
        document.createElement("div");

    right.style.width = "40%";
    right.style.display = "flex";
    right.style.flexDirection = "column";
    right.style.gap = "8px";

    main.appendChild(right);

    // =====================================================
    // CREATE CARD
    // =====================================================

    function createCard(titleText,height=260) {

        const card =
            document.createElement("div");

        card.style.background = "white";
        card.style.padding = "6px";
        card.style.borderRadius = "6px";

        const title =
            document.createElement("div");

        title.innerText = titleText;

        title.style.fontWeight = "bold";
        title.style.marginBottom = "4px";

        card.appendChild(title);

        const svg =
            document.createElementNS(
                "http://www.w3.org/2000/svg",
                "svg"
            );

        svg.setAttribute("width","620");
        svg.setAttribute("height",height);

        svg.style.border =
            "1px solid #dddddd";

        card.appendChild(svg);

        return {
            card,
            svg
        };
    }

    // =====================================================
    // RESULT PLOTS
    // =====================================================

    const resultPlot =
        createCard(
            "Señal resultante",
            260
        );

    right.appendChild(
        resultPlot.card
    );

    const fftPlot =
        createCard(
            "FFT",
            260
        );

    right.appendChild(
        fftPlot.card
    );

    const stftPlot =
        createCard(
            "STFT Spectrogram",
            320
        );

    right.appendChild(
        stftPlot.card
    );

    // =====================================================
    // INPUT
    // =====================================================

    function createInput(label,value) {

        const c =
            document.createElement("div");

        c.style.display = "flex";
        c.style.flexDirection = "column";
        c.style.marginBottom = "6px";

        const l =
            document.createElement("label");

        l.innerText = label;

        l.style.fontSize = "12px";

        const i =
            document.createElement("input");

        i.type = "number";
        i.value = value;

        i.style.padding = "4px";

        c.appendChild(l);
        c.appendChild(i);

        return {
            container:c,
            input:i
        };
    }

    // =====================================================
    // SLIDER
    // =====================================================

    function createSlider(
        label,
        value,
        minVal=-5,
        maxVal=5
    ) {

        const c =
            document.createElement("div");

        c.style.display = "flex";
        c.style.flexDirection = "column";
        c.style.marginBottom = "6px";

        const row =
            document.createElement("div");

        row.style.display = "flex";

        row.style.justifyContent =
            "space-between";

        const l =
            document.createElement("label");

        l.innerText = label;

        const valueText =
            document.createElement("span");

        valueText.innerText =
            parseFloat(value).toFixed(1);

        valueText.style.fontWeight = "bold";
        valueText.style.color = "#0066ff";

        row.appendChild(l);
        row.appendChild(valueText);

        const s =
            document.createElement("input");

        s.type = "range";

        s.min = minVal;
        s.max = maxVal;
        s.step = 0.1;
        s.value = value;

        s.addEventListener("input",()=>{

            valueText.innerText =
                parseFloat(s.value).toFixed(1);
        });

        c.appendChild(row);
        c.appendChild(s);

        return {
            container:c,
            slider:s
        };
    }

    // =====================================================
    // GRID
    // =====================================================

    function drawGrid(
        svg,
        w,
        h
    ) {

        svg.innerHTML = "";

        for(let x=50;x<w;x+=40) {

            const line =
                document.createElementNS(
                    "http://www.w3.org/2000/svg",
                    "line"
                );

            line.setAttribute("x1",x);
            line.setAttribute("y1",10);

            line.setAttribute("x2",x);
            line.setAttribute("y2",h-30);

            line.setAttribute(
                "stroke",
                "#ededed"
            );

            svg.appendChild(line);
        }

        for(let y=20;y<h-30;y+=40) {

            const line =
                document.createElementNS(
                    "http://www.w3.org/2000/svg",
                    "line"
                );

            line.setAttribute("x1",50);
            line.setAttribute("y1",y);

            line.setAttribute("x2",w);
            line.setAttribute("y2",y);

            line.setAttribute(
                "stroke",
                "#ededed"
            );

            svg.appendChild(line);
        }

        const xAxis =
            document.createElementNS(
                "http://www.w3.org/2000/svg",
                "line"
            );

        xAxis.setAttribute("x1",50);
        xAxis.setAttribute("y1",h/2);

        xAxis.setAttribute("x2",w);
        xAxis.setAttribute("y2",h/2);

        xAxis.setAttribute(
            "stroke",
            "#666"
        );

        svg.appendChild(xAxis);

        const yAxis =
            document.createElementNS(
                "http://www.w3.org/2000/svg",
                "line"
            );

        yAxis.setAttribute("x1",50);
        yAxis.setAttribute("y1",10);

        yAxis.setAttribute("x2",50);
        yAxis.setAttribute("y2",h-30);

        yAxis.setAttribute(
            "stroke",
            "#666"
        );

        svg.appendChild(yAxis);
    }

    // =====================================================
    // DRAW SIGNAL
    // =====================================================

    function drawSignal(
        svg,
        signal,
        color,
        totalTime
    ) {

        const w =
            parseInt(
                svg.getAttribute("width")
            );

        const h =
            parseInt(
                svg.getAttribute("height")
            );

        drawGrid(svg,w,h);

        let peak =
            Math.max(
                ...signal.map(
                    v=>Math.abs(v)
                )
            );

        if(peak < 0.001)
            peak = 1;

        const scale =
            (h*0.35)/peak;

        let path = "";

        for(let i=0;i<signal.length;i++) {

            const x =
                50 +
                i*((w-70)/signal.length);

            const y =
                h/2 -
                signal[i]*scale;

            if(i===0) {

                path += `M ${x} ${y}`;

            } else {

                path += ` L ${x} ${y}`;
            }
        }

        const curve =
            document.createElementNS(
                "http://www.w3.org/2000/svg",
                "path"
            );

        curve.setAttribute("d",path);

        curve.setAttribute(
            "stroke",
            color
        );

        curve.setAttribute(
            "stroke-width",
            "2"
        );

        curve.setAttribute(
            "fill",
            "none"
        );

        svg.appendChild(curve);

        const txt =
            document.createElementNS(
                "http://www.w3.org/2000/svg",
                "text"
            );

        txt.setAttribute("x",w-140);

        txt.setAttribute("y","18");

        txt.setAttribute(
            "font-size",
            "11"
        );

        txt.textContent =
            "Peak: ±" +
            peak.toFixed(2);

        svg.appendChild(txt);

        for(let i=0;i<=10;i++) {

            const t =
                i * totalTime / 10;

            const x =
                50 +
                i*((w-70)/10);

            const txt =
                document.createElementNS(
                    "http://www.w3.org/2000/svg",
                    "text"
                );

            txt.setAttribute("x",x-8);

            txt.setAttribute("y",h-12);

            txt.setAttribute(
                "font-size",
                "10"
            );

            txt.textContent =
                t.toFixed(1);

            svg.appendChild(txt);
        }
    }

"""
    
    @property
    def signal_numpy(self):

        return np.array(self.signals[0])


    @property
    def signals_numpy(self):

        return [
            np.array(s)
            for s in self.signals
        ]


ui = SignalDraw()

ui